# QDev Crew Adapter — LoRA & QLoRA

One adapter that teaches an open 7B QDev's house language across all seven SDLC
tasks — **plan · dev · review · qe · bug · devops · custom** — so every agent in
the platform, including user-built custom agents, speaks the same structured
contract.

**Base model:** `HuggingFaceH4/zephyr-7b-beta` (Mistral-7B, 32k vocab).
**Runtime → Change runtime type → L4 GPU.**

Buying Colab Pro does not change the runtime type on its own, and it does **not**
include background execution — that is Pro+. Cell 0 checks the GPU; the script
refuses to start on a Turing card rather than quietly producing an adapter that
cannot do QE.

Because the tab has to stay open, **both long cells are resumable** — see cell 2b.
After any disconnect, re-run cells 1, 2, 2b and then the cell that died.

## Timing — measured on this L4

| # | Cell | time |
|---|---|---|
| 0–2 | GPU check, install, upload | ~5 min |
| 2b | **Drive** — makes everything resumable | ~1 min |
| 3 | smoke (incl. ~15 GB model download) | ~15 min |
| 4 | **baseline eval**, 74 rows | ~75 min |
| 5 | **qlora train** — 3 epochs, 7.7M tokens | **~7.7 h** |
| 6 | **qlora eval** | ~75 min |
| 7 | lora train (bf16 arm) | ~5 h |
| 8 | lora eval | ~75 min |
| 9 | download | ~1 min |

Throughput is measured, not guessed: **277 training tokens/s**, ~20 decode
tokens/s in bf16. `--epochs 2` on cell 5 cuts training to ~5.1 h if you want it
finished sooner.

**Run the cells with no extra flags.** The defaults are the full-quality ones
(74 eval rows, batch 4, window auto-sized from measured VRAM). An older copy of
this notebook passed `--per-task 3 --batch 2`, which silently re-imposes the
small-GPU compromise and leaves per-task numbers too noisy to publish.

## Why the vocabulary picked the model

On this card the binding constraint is the logits tensor — `vocab × seq`, held
about three times over during a backward pass:

| base | vocab | logits @6144 | outcome |
|---|---|---|---|
| Qwen2.5-7B | 152,064 | 7.0 GB | OOM even on an L4 window of 12288 |
| **zephyr-7b-beta** | **32,000** | **1.5 GB** | fits at 12288, 100% of the corpus |

Same 7B class, ungated, 4.75× smaller vocabulary. And zephyr is Mistral-7B
architecture, which is what `calc/report.py` already models — so the run reports
**41,943,040 trainable parameters** and the LinkedIn arithmetic says the same.

At a 4096 window only 1 of 120 `qe` training examples still fits its target, so
the window is not a speed knob — it decides whether Velma's task exists at all.

## Baseline expectations

From the first (39-row) baseline: **overall 0.507**, with `plan` already at
0.916 and `qe` at 0.135. So the story is not "everything improved" — it is
"`qe`, `bug` and `review` went from broken to working, `plan` was already fine."
Lead with the per-task chart, not the overall number.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# ── CHECK THIS BEFORE GOING FURTHER ──────────────────────────────────────
#   WANT:  NVIDIA L4, 23034 MiB, 8.9     (or A100, 40960 MiB, 8.0)
#   WRONG: Tesla T4, 15360 MiB, 7.5
#
# Buying Colab Pro does NOT change your runtime type. If this says Tesla T4,
# go to Runtime -> Change runtime type -> L4 GPU -> Save. That restarts the
# runtime, so re-run this cell and the two below it afterwards.
#
# It is not a speed preference. Turing (7.5) has no flash-attention kernel, so
# attention memory is heads x seq^2 rather than linear, which caps the window
# at 4096 — and at 4096 only 1 of 120 qe training examples still fits its
# target. The adapter would never learn to author a test plan.

In [ ]:
%pip -q install -U "transformers>=4.57" "peft>=0.17" "trl>=0.24" \
  "bitsandbytes>=0.48" "accelerate>=1.10" "datasets>=3.0"
# liger-kernel is optional now — its fused cross-entropy matters for large-vocab
# models, and zephyr's 32k vocabulary makes the logits tensor small enough that
# it isn't needed. Uncomment to try it; the script probes it and falls back.
# %pip -q install -U "liger-kernel>=0.5"
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 2 · Upload the bundle

Locally, run `python -m colab.package` to produce `qdev_finetune_bundle.zip`
(3.2 MB), then upload it here. It carries `run_finetune.py`, the `evaluate/`
scorer and the four dataset splits.

The scorer travels *with* the data on purpose — the before/after comparison only
means something if both runs are graded by the identical function, so it ships
alongside rather than being re-implemented in a cell.

In [ ]:
from google.colab import files
import zipfile, pathlib, os, json
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z: z.extractall('.')
man = json.load(open('MANIFEST.json'))
for s, v in man['splits'].items():
    print(f"{s:<10}{v['n']:>5}   " + ' '.join(f'{k}={n}' for k, n in sorted(v['per_task'].items())))
# The hashes are here so that if a Colab number ever disagrees with a local one,
# "were the same bytes graded?" is answerable instead of debatable.
print('\nscorer sha256:', man['files']['evaluate/scorer.py']['sha256'])

## 2b · Persist `out/` to Drive — do not skip this

Colab **Pro does not include background execution** (that is Pro+). The run
depends on a browser tab staying open for up to 8 hours, so treat a disconnect
as likely rather than exceptional.

This points `out/` at Google Drive, which makes both long cells resumable:

- **evals** checkpoint after every batch. Re-run the same cell and it replays
  the finished rows from disk instead of regenerating them. Greedy decoding
  makes that exact — a resumed eval and an uninterrupted one produce identical
  numbers.
- **training** saves every 10 steps (~30 min). Re-run the cell and it continues
  from the newest checkpoint rather than step 0.

So after any disconnect: re-run cells 1, 2, 2b, then **the same cell that died**.

### Keeping the tab alive

- Stop the machine sleeping — Windows: Settings → System → Power → Screen and
  sleep → *Never* (on the "plugged in" setting)
- Leave the Colab tab open; a background tab is fine, a closed one is not
- Do not let the laptop lid close

The base model is **not** cached to Drive: zephyr is ~15 GB and free Drive is 15
GB total. A fresh runtime re-downloads it in ~40 s on an L4.

In [ ]:
import os, json, pathlib
from google.colab import drive
drive.mount('/content/drive')

DEST = pathlib.Path('/content/drive/MyDrive/qdev_finetune/out')
DEST.mkdir(parents=True, exist_ok=True)

# run_finetune.py writes to the relative path "out". Symlinking it means the
# script needs no Drive awareness at all — and `mkdir(exist_ok=True)` on a
# symlink pointing at a live directory is a no-op, so nothing breaks.
if pathlib.Path('out').is_symlink():
    os.unlink('out')
elif pathlib.Path('out').exists():
    raise SystemExit("a real ./out already exists — move it before symlinking, "
                     "or you will lose whatever is in it")
os.symlink(DEST, 'out')

res = DEST / 'results.json'
done = sorted(json.load(res.open()).keys()) if res.exists() else []
print(f"out/ -> {DEST}")
print("stages already recorded:", ', '.join(done) if done else "none (fresh start)")
print("adapters on Drive:", [p.name for p in DEST.iterdir() if p.is_dir()] or "none")

## 3 · Smoke test — run this before anything expensive

Four training steps on the **eight longest examples in the corpus** (peak memory
is set by the longest sequence, so a smoke test on average rows proves nothing),
then one eval row per task at a 1200-token cap.

Includes a one-time ~15 GB model download.

### Reading the result

```
GPU NVIDIA L4  sm_89  22.0 GiB
seq 12288: needs 7.1 GiB (attn 0.2 + acts 3.0 + logits 2.9 + grads/opt 1.0)  fits
seq window 12288
trainable 41,943,040 / ... = 1.11%
```

`attn 0.2` is flash attention — on a T4 that same line reads `attn 18.0`.
`12288` means no training example was dropped. `41,943,040` matches
`calc/report.py`.

Then the per-task eval table. **`bug`, `review` and `plan` should show
`emit` above 0** — their objects fit inside 1200 tokens, so a non-zero emit
proves the chat template, generation and JSON extraction all work. `qe` and
`dev` will truncate and read 0.00; that is the cap, not a fault.

If **every** task shows `emit 0.00`, the script says so explicitly and points at
`out/raw_*.json`. Truncated JSON there is fine. Prose or empty strings are not —
stop and investigate rather than spending three hours measuring a broken harness.

In [ ]:
# Saves to out/smoke, NOT out/qlora — so a half-trained 4-step adapter can never
# be mistaken for the real one if a later cell fails.
!python run_finetune.py --stage qlora --smoke --name smoke && \
 python run_finetune.py --stage eval --adapter out/smoke --name wiring --smoke --batch 2 && \
 echo "SMOKE OK — the long cells will run"

## 4 · Baseline — the untouched base model

This is the ***before*** number, and it is the one that has to be honest. Nothing
has been trained yet: same 74 eval rows, same scorer, same greedy decoding, same
generation ceilings as every later stage.

**18 of the 74 rows are real production artifacts** — stories a human wrote and
Claude answered inside QDev — replayed through the exact production prompt
builders (`manager_agent._build_user_prompt`, `qe_agent._build_user_prompt`). The
adapter never sees them in training. The other 56 cover the five tasks that have
no production data yet.

Expect it to be low. A base instruct model handed a strict JSON contract usually
writes prose, wraps the object in commentary, or invents field names. That gap is
the story.

This is the slowest cell, because a base model rarely emits EOS and so runs to
the generation ceiling on most rows. The tuned model will be much faster.

In [ ]:
# No flags: the defaults are now the full-quality ones (74 eval rows, batch 4,
# window auto-sized from measured VRAM). Passing --per-task 3 here would silently
# re-impose the T4 compromise.
!python run_finetune.py --stage baseline

## 5 · QLoRA training

4-bit NF4 base + double quantization + paged 8-bit AdamW — the three techniques
that make a 7B trainable on one card. **3 epochs over the full 827-example
corpus**, all seven tasks, nothing dropped.

Watch the `trainable / total` line. That percentage is the LoRA argument in one
number, and it matches what `calc/report.py` derives locally from Mistral-7B's
real dimensions: **41,943,040 trainable parameters**.

In [ ]:
!python run_finetune.py --stage qlora --rank 16 --alpha 32
# rank 16 / alpha 32 is deliberate, not a default left alone: it is the exact
# configuration calc/report.py models, so the adapter this produces is the one
# the LinkedIn arithmetic describes. epochs defaults to 3.

In [ ]:
!python run_finetune.py --stage eval --adapter out/qlora --name qlora
# 6 · The AFTER number. Identical eval set, scorer, ceilings and greedy decoding
# as the baseline — that identity is the only reason the comparison means
# anything.

## 7 · LoRA training — the comparison arm

fp16 base instead of NF4. Same rank, same data, same schedule, so the *only*
variable is whether the frozen base is quantized. That is what makes it a fair
LoRA-vs-QLoRA comparison rather than two unrelated runs.

On a T4 this was hopeless — a 7B in fp16 is ~14.5 GB of weights before a single
activation. On a 22.5 GB L4 it fits, which is why this arm is worth running now:
you get the real answer (does 4-bit quantization of the frozen base cost any
quality?) instead of an OOM you have to explain away.

In [ ]:
!python run_finetune.py --stage lora --rank 16 --alpha 32

In [ ]:
!python run_finetune.py --stage eval --adapter out/lora --name lora

## 9 · Download results

`results.json` drives the charts. `raw_*.json` holds the actual model responses,
which is what makes the before/after side-by-side possible — send both back.

In [ ]:
import json, shutil
res = json.load(open('out/results.json'))
# smoke_* rows are wiring checks at a 400-token ceiling — never report them.
real = {k: v for k, v in res.items() if not k.startswith('smoke_')}
print(f"{'run':<12}{'score':>8}{'emit':>8}{'schema':>8}{'enums':>8}{'rigor':>8}{'prod':>8}")
for run, v in real.items():
    o = v['overall']; r = v.get('by_slice', {}).get('real', {})
    print(f"{run:<12}{o['score']:>8.3f}{o['emit_rate']:>8.3f}{o['schema']:>8.3f}"
          f"{o['enums']:>8.3f}{o['rigor']:>8.3f}{r.get('score', float('nan')):>8.3f}")
if 'baseline' in real and 'qlora' in real:
    b, q = real['baseline']['overall']['score'], real['qlora']['overall']['score']
    print(f"\nbefore {b:.3f} -> after {q:.3f}   "
          f"+{q-b:.3f} absolute, {q/max(b, 1e-9):.1f}x")
shutil.make_archive('qdev_results', 'zip', 'out')
from google.colab import files; files.download('qdev_results.zip')